# Sample equifax / test  (target-first 400k -> app -> aggregate NEW + OLD)

1. Sample from `equifax/test` **target**, keeping only rows with a **non-null** target, down to **400,000** applicants; grab their `ZEST_KEY`s.
2. Save `target.parquet` and the matching `app.parquet` at the top.
3. Aggregate **only those ZEST_KEYs** for **both** `processed_new` and `processed_old` (ZEST_KEY kept as the index).

Outputs to `samples/equifax_test/`.

Run in the model-engine kernel.

In [1]:
1+1

2

In [2]:
import os, sys, gc, glob, shutil, warnings
import pandas as pd
import pyarrow as pa
import pyarrow.dataset as pds

for cand in [os.path.abspath(os.path.join(os.getcwd(), '..')), os.getcwd(),
             '/home/jag/payment-processor-research']:
    if os.path.exists(os.path.join(cand, 'configs.py')):
        sys.path.insert(0, cand); break
from configs import DATA_DIR

from model_engine.assets.utils import load_asset
from model_engine.feature_engine_V2.feature_engine import AggregationEngine
warnings.filterwarnings('ignore')

BUREAU, ROLE, SOURCE_SPLIT = 'equifax', 'test', 'test'
N_ROWS     = 400_000
SEED       = 42
TARGET_COL = 'final_DQ60_m24'   # non-null filter target (set to 'final_DQ60_m30' if that's what you meant)

OUT = os.path.join(DATA_DIR, 'samples', f'{BUREAU}_{ROLE}')
os.makedirs(OUT, exist_ok=True)
agg_eng = AggregationEngine(asset=load_asset('aggregation/fe2/trade.json'), table_name='trade')
print(f'{BUREAU}/{ROLE}  (source split = {SOURCE_SPLIT})  ->  {OUT}')

equifax/test  (source split = test)  ->  /home/jag/payment-processor-research/payment_processing_research_data/samples/equifax_test


## Why we wipe `samples/<bureau>_<role>/` first

The next cell deletes this combo's whole sample folder before writing anything. This keeps each run
**fully reproducible and idempotent**:

- a previous run may have used a different `N_ROWS`, `SEED`, or `TARGET_COL` — leftover `app.parquet` /
  `target.parquet` / `processed_*` parts would otherwise be a stale mix of two different samples.
- the older **app-first** sampling wrote different rows here; we don't want those bleeding into the new
  target-first sample.
- each notebook owns exactly **one** folder (`samples/<bureau>_<role>/`), so deleting and rebuilding it
  is safe — it never touches another bureau/role.

Net effect: re-running a notebook always yields a clean, self-consistent `app` + `target` + `processed_new` + `processed_old` for the same 400k ZEST_KEYs.

In [3]:
# wipe any previous sample for this combo so nothing stale lingers
if os.path.isdir(OUT):
    shutil.rmtree(OUT)
os.makedirs(OUT, exist_ok=True)

# 1) sample from TARGET first -- keep only applicants with a non-null target, then sample N_ROWS
tgt = pd.read_parquet(os.path.join(DATA_DIR, BUREAU, SOURCE_SPLIT, 'target'))
tgt = tgt[tgt[TARGET_COL].notna()].drop_duplicates('ZEST_KEY')
n   = min(N_ROWS, len(tgt))
if n < N_ROWS:
    print(f'WARNING: only {len(tgt):,} non-null {TARGET_COL} target rows (< {N_ROWS:,})')
tgt_s = tgt.sample(n=n, random_state=SEED).copy()
keys  = set(tgt_s['ZEST_KEY'])

# 2) app for exactly those ZEST_KEYs
app   = pd.read_parquet(os.path.join(DATA_DIR, BUREAU, SOURCE_SPLIT, 'app')).drop_duplicates('ZEST_KEY')
app_s = app[app['ZEST_KEY'].isin(keys)].copy()

# 3) save app + target at the top
tgt_s.to_parquet(os.path.join(OUT, 'target.parquet'), index=False)
app_s.to_parquet(os.path.join(OUT, 'app.parquet'),    index=False)
print(f'non-null {TARGET_COL}: target sampled {len(tgt_s):,} | app matched {len(app_s):,} | unique keys {len(keys):,}')

non-null final_DQ60_m24: target sampled 400,000 | app matched 400,000 | unique keys 400,000


In [4]:
# 4) aggregate ONLY the sampled ZEST_KEYs, for BOTH variants, in ONE pass each
#    (1.5 TB RAM here, so the ~28 GB output frame is fine -- no bucketing needed).
#    pyarrow predicate pushdown grabs only the sampled keys' rows off disk.
key_arr = pa.array(list(keys))

def aggregate_variant(variant):
    norm_dir = os.path.join(DATA_DIR, BUREAU, SOURCE_SPLIT, f'normalized_{variant}')
    assert glob.glob(os.path.join(norm_dir, 'part-*.parquet')), f'no normalized_{variant} at {norm_dir}'
    proc_dir = os.path.join(OUT, f'processed_{variant}')
    if os.path.isdir(proc_dir):
        shutil.rmtree(proc_dir)
    os.makedirs(proc_dir, exist_ok=True)

    print(f'[{variant}] grabbing normalized rows for the sampled keys ...')
    norm = (pds.dataset(norm_dir, format='parquet')
               .to_table(filter=pds.field('ZEST_KEY').isin(key_arr))
               .to_pandas())
    print(f'[{variant}] got {len(norm):,} normalized rows -- starting aggregation ...')
    processed = agg_eng.transform(norm)        # all at once
    processed.index.name = 'ZEST_KEY'
    processed.to_parquet(os.path.join(proc_dir, 'part-000.parquet'), index=True)
    print(f'[{variant}] done: {len(processed):,} ZEST_KEYs x {processed.shape[1]} cols -> {proc_dir}')
    del norm, processed; gc.collect()

for v in ['new', 'old']:
    aggregate_variant(v)

[new] grabbing normalized rows for the sampled keys ...
[new] got 8,101,322 normalized rows -- starting aggregation ...
[new] done: 393,722 ZEST_KEYs x 8848 cols -> /home/jag/payment-processor-research/payment_processing_research_data/samples/equifax_test/processed_new
[old] grabbing normalized rows for the sampled keys ...
[old] got 8,101,322 normalized rows -- starting aggregation ...
[old] done: 393,722 ZEST_KEYs x 8848 cols -> /home/jag/payment-processor-research/payment_processing_research_data/samples/equifax_test/processed_old


In [5]:
# 5) checks
for v in ['new', 'old']:
    df = pd.read_parquet(os.path.join(OUT, f'processed_{v}'))
    assert df.index.name == 'ZEST_KEY', f'{v}: ZEST_KEY not the index ({df.index.name!r})'
    print(f'processed_{v}: {len(df):,} rows, ZEST_KEY index, {df.shape[1]} feature cols, unique {df.index.nunique():,}')
app_n = len(pd.read_parquet(os.path.join(OUT, 'app.parquet')))
tgt_n = len(pd.read_parquet(os.path.join(OUT, 'target.parquet')))
print(f'app {app_n:,} | target {tgt_n:,}')
print('DONE', BUREAU, ROLE)

processed_new: 393,722 rows, ZEST_KEY index, 8848 feature cols, unique 393,722
processed_old: 393,722 rows, ZEST_KEY index, 8848 feature cols, unique 393,722
app 400,000 | target 400,000
DONE equifax test
